# Consent-Based Teacher Voice Adaptation

This notebook prepares a pretrained text-to-speech model for a consented teacher voice adaptation workflow.

What this notebook does:
- Validates consented speech recordings and transcripts
- Loads a pretrained TTS base model
- Applies a lightweight LoRA adapter for CPU-friendly fine-tuning
- Extracts a reference speaker embedding from consented audio
- Saves adapter and configuration artifacts for later inference

What this notebook does not do:
- Train a voice model from scratch
- Work without a consented dataset

Expected inputs:
- `voice_data/manifest.jsonl` with `audio_path` and `text` fields
- Clean WAV recordings of the teacher with matching transcripts

Expected outputs:
- `voice_model_artifacts/teacher_voice_lora_adapter`
- `voice_model_artifacts/teacher_voice_package.pt`
- `voice_model_artifacts/train_config.json`

In [38]:
import gc
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

In [39]:
import os
os.getcwd()
os.chdir('/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project')
os.getcwd()

'/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project'

## presets

In [40]:
# Paths and artifact names
PROJECT_ROOT_DEFAULT = "."
MANIFEST_REL_PATH = "voice_data/manifest.jsonl"
OUTPUT_DIR_NAME = "voice_model_artifacts"
ADAPTER_DIR_NAME = "teacher_voice_lora_adapter"
PACKAGE_FILENAME = "teacher_voice_package.pt"
TRAIN_CONFIG_FILENAME = "train_config.json"
PROCESSOR_DIR_NAME = "processor"
VOCODER_DIR_NAME = "vocoder"

# Model and runtime defaults
BASE_MODEL_NAME = "microsoft/speecht5_tts"
VOCODER_MODEL_NAME = "microsoft/speecht5_hifigan"
SPEAKER_ENCODER_NAME = "speechbrain/spkrec-ecapa-voxceleb"
DEVICE_NAME = "cpu"
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Reused numeric constants
MFCC_DIMENSIONS = 40
NORM_EPSILON = 1e-8
MAX_REFERENCE_ROWS = 5

## configuration

In [41]:
@dataclass
class VoiceAdaptConfig:
    project_root: str = PROJECT_ROOT_DEFAULT
    manifest_path: str = MANIFEST_REL_PATH
    output_dir: str = OUTPUT_DIR_NAME

    base_model_name: str = BASE_MODEL_NAME
    vocoder_name: str = VOCODER_MODEL_NAME
    speaker_encoder_name: str = SPEAKER_ENCODER_NAME

    sample_rate: int = 16000
    max_audio_seconds: float = 8.0
    max_text_length: int = 256

    batch_size: int = 1
    grad_accum_steps: int = 4
    learning_rate: float = 1e-4
    num_epochs: int = 3
    seed: int = 42
    cpu_threads: int = 4

    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05


cfg = VoiceAdaptConfig()
PROJECT_ROOT = Path(cfg.project_root).resolve()
MANIFEST_PATH = (PROJECT_ROOT / cfg.manifest_path).resolve()
OUTPUT_DIR = (PROJECT_ROOT / cfg.output_dir).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(DEVICE_NAME)
torch.set_num_threads(cfg.cpu_threads)
torch.manual_seed(cfg.seed)
random.seed(cfg.seed)

{
    "manifest": str(MANIFEST_PATH),
    "output_dir": str(OUTPUT_DIR),
    "device": str(DEVICE),
    "base_model": cfg.base_model_name,
    "vocoder": cfg.vocoder_name,
}

{'manifest': '/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project/voice_data/manifest.jsonl',
 'output_dir': '/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project/voice_model_artifacts',
 'device': 'cpu',
 'base_model': 'microsoft/speecht5_tts',
 'vocoder': 'microsoft/speecht5_hifigan'}

## functions

### normalize_text
Standardizes transcript text by trimming whitespace, lowering case, and collapsing duplicate spaces.

In [42]:
def normalize_text(text: str) -> str:
    return " ".join(text.strip().lower().split())

### load_manifest
Reads and validates the JSONL manifest, resolves audio paths, and normalizes transcript text.

In [43]:
def load_manifest(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(
            f"Manifest not found at {path}. Create JSONL with keys: audio_path, text"
        )

    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            if "audio_path" not in item or "text" not in item:
                raise ValueError("Each manifest row needs 'audio_path' and 'text'.")

            audio_path = Path(item["audio_path"])
            resolved_audio = (
                str((path.parent / audio_path).resolve())
                if not audio_path.is_absolute()
                else str(audio_path)
            )

            rows.append(
                {
                    "audio_path": resolved_audio,
                    "text": normalize_text(item["text"]),
                }
            )

    if not rows:
        raise ValueError("Manifest is empty.")

    return rows

### build_dataset_summary
Computes quick metadata about the loaded training rows for sanity checking.

In [44]:
def build_dataset_summary(rows: list[dict[str, str]]) -> dict[str, Any]:
    return {
        "num_examples": len(rows),
        "avg_text_length": float(np.mean([len(row["text"]) for row in rows])) if rows else 0.0,
        "sample_audio_paths": [row["audio_path"] for row in rows[:3]],
    }

### load_audio_mono
Loads mono audio at a fixed sample rate and trims it to a configured maximum duration.

In [45]:
def load_audio_mono(audio_path: Path, sample_rate: int, max_audio_seconds: float) -> np.ndarray:
    waveform, _ = librosa.load(audio_path, sr=sample_rate, mono=True)
    max_samples = int(sample_rate * max_audio_seconds)
    return waveform[:max_samples]

### TeacherVoiceDataset
Wraps manifest rows as dataset items with waveform tensors and metadata.

In [46]:
class TeacherVoiceDataset(Dataset):
    def __init__(self, rows: list[dict[str, str]], cfg: VoiceAdaptConfig):
        self.rows = rows
        self.cfg = cfg

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.rows[idx]
        audio_path = Path(row["audio_path"])
        waveform = load_audio_mono(audio_path, self.cfg.sample_rate, self.cfg.max_audio_seconds)

        return {
            "text": row["text"],
            "audio_path": str(audio_path),
            "waveform": torch.tensor(waveform, dtype=torch.float32),
            "duration_seconds": float(waveform.shape[0] / self.cfg.sample_rate),
        }

### collate_teacher_batch
Pass-through collate function for dataloader compatibility.

In [47]:
def collate_teacher_batch(batch: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return batch

### load_pretrained_stack
Loads the processor, base SpeechT5 model, and vocoder from configured model names.

In [48]:
def load_pretrained_stack(cfg: VoiceAdaptConfig):
    processor = SpeechT5Processor.from_pretrained(cfg.base_model_name)
    model = SpeechT5ForTextToSpeech.from_pretrained(cfg.base_model_name)
    vocoder = SpeechT5HifiGan.from_pretrained(cfg.vocoder_name)
    return processor, model, vocoder

### apply_lora_to_model
Attaches and freezes a LoRA adapter so only adapter weights are trained.

In [49]:
def apply_lora_to_model(model: SpeechT5ForTextToSpeech, cfg: VoiceAdaptConfig):
    lora_config = LoraConfig(
        r=cfg.lora_rank,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)
    for name, param in model.named_parameters():
        if "lora_" not in name:
            param.requires_grad = False
    return model

### _extract_mfcc_embedding
Builds a lightweight speaker embedding from MFCC statistics.

In [50]:
def _extract_mfcc_embedding(audio_path: str, cfg: VoiceAdaptConfig) -> torch.Tensor:
    """Create a lightweight CPU speaker representation from MFCC statistics."""
    waveform = load_audio_mono(Path(audio_path), cfg.sample_rate, cfg.max_audio_seconds)
    mfcc = librosa.feature.mfcc(
        y=waveform,
        sr=cfg.sample_rate,
        n_mfcc=MFCC_DIMENSIONS,
    )
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    feat = np.concatenate(
        [
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            delta.mean(axis=1),
            delta.std(axis=1),
            delta2.mean(axis=1),
            delta2.std(axis=1),
        ]
    )
    embedding = torch.tensor(feat, dtype=torch.float32)
    embedding = embedding / embedding.norm(p=2).clamp(min=NORM_EPSILON)
    return embedding

### extract_speaker_embedding
Aggregates per-recording embeddings into one normalized speaker embedding.

In [51]:
def extract_speaker_embedding(reference_rows: list[dict[str, str]], cfg: VoiceAdaptConfig) -> torch.Tensor:
    if not reference_rows:
        raise ValueError("At least one reference recording is required for the speaker embedding.")

    embeddings = [_extract_mfcc_embedding(row["audio_path"], cfg) for row in reference_rows]
    speaker_embedding = torch.stack(embeddings).mean(dim=0)
    speaker_embedding = speaker_embedding / speaker_embedding.norm(p=2).clamp(min=NORM_EPSILON)
    return speaker_embedding

### prepare_processor_batch
Tokenizes text/audio into tensor-only model inputs.

In [52]:
def prepare_processor_batch(processor: SpeechT5Processor, text: str, waveform: np.ndarray, cfg: VoiceAdaptConfig) -> dict[str, torch.Tensor]:
    batch = processor(text=text, audio=waveform, sampling_rate=cfg.sample_rate, return_tensors="pt")
    return {key: value for key, value in batch.items() if isinstance(value, torch.Tensor)}

### train_lora_adapter
Runs LoRA fine-tuning and records loss history by epoch.

In [53]:
def train_lora_adapter(
    model: SpeechT5ForTextToSpeech,
    processor: SpeechT5Processor,
    dataset: TeacherVoiceDataset,
    speaker_embedding: torch.Tensor,
    cfg: VoiceAdaptConfig,
) -> list[dict[str, float]]:
    model.train()
    optimizer = torch.optim.AdamW(
        [param for param in model.parameters() if param.requires_grad],
        lr=cfg.learning_rate,
    )

    history: list[dict[str, float]] = []
    for epoch in range(1, cfg.num_epochs + 1):
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, item in enumerate(dataset, start=1):
            features = prepare_processor_batch(
                processor=processor,
                text=item["text"],
                waveform=item["waveform"].numpy(),
                cfg=cfg,
            )
            features = {key: value.to(DEVICE) for key, value in features.items()}
            speaker_batch = speaker_embedding.to(DEVICE).unsqueeze(0)

            outputs = model(**features, speaker_embeddings=speaker_batch)
            loss = outputs.loss
            (loss / cfg.grad_accum_steps).backward()
            running_loss += float(loss.detach().cpu().item())

            if step % cfg.grad_accum_steps == 0 or step == len(dataset):
                torch.nn.utils.clip_grad_norm_(
                    [param for param in model.parameters() if param.requires_grad],
                    max_norm=1.0,
                )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        gc.collect()
        history.append({"epoch": float(epoch), "loss": running_loss / max(len(dataset), 1)})

    return history

### save_voice_artifacts
Writes the trained adapter, processor/vocoder assets, and config package to disk.

In [54]:
def save_voice_artifacts(
    model: SpeechT5ForTextToSpeech,
    processor: SpeechT5Processor,
    vocoder: SpeechT5HifiGan,
    speaker_embedding: torch.Tensor,
    cfg: VoiceAdaptConfig,
    history: list[dict[str, float]],
) -> dict[str, str]:
    adapter_dir = OUTPUT_DIR / ADAPTER_DIR_NAME
    package_path = OUTPUT_DIR / PACKAGE_FILENAME
    config_path = OUTPUT_DIR / TRAIN_CONFIG_FILENAME

    adapter_dir.mkdir(parents=True, exist_ok=True)
    processor.save_pretrained(str(OUTPUT_DIR / PROCESSOR_DIR_NAME))
    vocoder.save_pretrained(str(OUTPUT_DIR / VOCODER_DIR_NAME))
    model.save_pretrained(str(adapter_dir))

    package = {
        "config": asdict(cfg),
        "history": history,
        "speaker_embedding": speaker_embedding.cpu(),
    }
    torch.save(package, package_path)

    with config_path.open("w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    return {
        "adapter_dir": str(adapter_dir),
        "package_path": str(package_path),
        "config_path": str(config_path),
    }

In [55]:
rows = load_manifest(MANIFEST_PATH)
summary = build_dataset_summary(rows)
dataset = TeacherVoiceDataset(rows=rows, cfg=cfg)

if len(dataset) == 0:
    raise ValueError("No training data found in the manifest.")

processor, model, vocoder = load_pretrained_stack(cfg)
model = apply_lora_to_model(model, cfg)

reference_rows = rows[: min(len(rows), MAX_REFERENCE_ROWS)]
speaker_embedding = extract_speaker_embedding(reference_rows=reference_rows, cfg=cfg)
history = train_lora_adapter(
    model=model,
    processor=processor,
    dataset=dataset,
    speaker_embedding=speaker_embedding,
    cfg=cfg,
)
artifacts = save_voice_artifacts(
    model=model,
    processor=processor,
    vocoder=vocoder,
    speaker_embedding=speaker_embedding,
    cfg=cfg,
    history=history,
)

{
    "dataset_summary": summary,
    "speaker_embedding_shape": tuple(speaker_embedding.shape),
    "artifacts": artifacts,
    "history": history,
}

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/585M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

AttributeError: 'SpeechT5ForTextToSpeech' object has no attribute 'prepare_inputs_for_generation'